In [1]:
import json
import os
import re
import openai
import pandas as pd
from pathlib import Path
from itertools import combinations
from collections import Counter
from dotenv import load_dotenv
from tqdm.auto import tqdm
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
)

### Helper Funcs and Constants

In [ ]:
ENV_PATH = Path("../.env")
DATA_DIR = Path("../data")
SRC_CSV = DATA_DIR / "combined_2017_2023_themes_with_KG.csv"
LEDGER_PATH = Path("synthetic_results.json")
FAILURES_PATH = Path("synthetic_failures.json")
OUT_CSV = DATA_DIR / "synthetic_golden_set.csv"
PAIRS_CSV = DATA_DIR / "golden_pairs.csv"
THEME_GOLD_CSV = DATA_DIR / "synthetic_gold_themes.csv"

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
LLM_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"
MAX_TOKENS = 8192

RUN_SIZE = 150
RANDOM_STATE = 72

ENTITY_TYPES = {
    "ORG",
    "ORG/GOV",
    "ORG/REG",
    "PERSON",
    "GPE",
    "COMP",
    "PRODUCT",
    "EVENT",
    "SECTOR",
    "ECON_INDICATOR",
    "FIN_INSTRUMENT",
    "CONCEPT",
}
THEME_RELATIONS = {"HAS_ACTOR", "HAS_TARGET", "HAS_CONTEXT", "AFFECTS"}
ENTITY_RELATIONS = {"ACTS_ON", "BELONGS_TO"}
ALL_RELATIONS = THEME_RELATIONS | ENTITY_RELATIONS
DIMENSIONS = {
    "economic_monetary_event",
    "geopolitical_factor",
    "sector_or_industry",
    "policy_or_regulation",
    "technology_concept",
    "macro_trend",
}

In [3]:
load_dotenv(ENV_PATH)
assert os.getenv("OPENROUTER_API_KEY"), "OPENROUTER_API_KEY missing from .env"

client = openai.OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=OPENROUTER_BASE_URL,
)

In [ ]:
SYSTEM_PROMPT = """You are a synthetic knowledge-graph generator used to build an
evaluation golden set. You are given ONE original news article's discovered
themes and its knowledge graph (KG). Produce reworded themes plus THREE
synthetic KGs (variants) over those themes, so that a good clustering system
would place them in the same cluster as the original.

═══════════════════════════════════════════════════
STEP 1 — REWORD THE THEMES (TWO PHRASINGS EACH)
═══════════════════════════════════════════════════
For EVERY original theme, write TWO new phrasings:

  near_paraphrase — almost the same phrase:
  • Minimal change: swap or reorder ONE word, or substitute one close synonym
  • Must NOT be the identical string (case-insensitive)
  • 2–5 words, a noun phrase, no company/person/place/product names, no numbers

  paraphrase — a genuine paraphrase:
  • Same meaning, clearly different wording (never identical or a trivial
    reorder; do not reuse the original or the near_paraphrase, case-insensitive)
  • 2–5 words, a noun phrase, no company/person/place/product names, no numbers

Both keep the SAME dimension as the original theme.
VARIANT 1 anchors to the near_paraphrase phrases; VARIANTS 2 and 3 anchor to
the paraphrase phrases.

═══════════════════════════════════════════════════
STEP 2 — PARTITION THE ORIGINAL ENTITIES (KEEP vs REPLACE)
═══════════════════════════════════════════════════
List the DISTINCT non-CONCEPT entities of the ORIGINAL KG; call the count N.
For EACH variant, split that list into a KEEP set and a REPLACE set:

  VARIANT 1 → KEEP ~85 - 90% of the N entities (replace at least ONE — never keep all)
  VARIANT 2 → KEEP ~70 - 80% of the N entities
  VARIANT 3 → KEEP ~40 - 50% of the N entities  (e.g. N=10 → keep 4 or 5)

KEEP means VERBATIM: a kept entity appears in the variant KG copied
character-for-character from the original KG — not renamed, not abbreviated,
not swapped for a competitor or analogue.

DO NOT write a parallel story about different companies. Every variant —
including VARIANT 3 — is still about mostly the SAME core actors as the
original article; ONLY the REPLACE-set entities change, each swapped for a
different plausible real-world entity of the SAME type.

Example with N=6 original entities
[nvidia, intel, graphics chips, data center business, wall street analysts,
nvidia revenue]:
  VARIANT 3 keeps 3 verbatim — kept_entities = ["nvidia", "graphics chips",
  "data center business"] — and replaces the other 3 (intel → amd, ...).
  An all-new cast (amd, broadcom, microsoft, ...) is WRONG: overlap with the
  original must stay ~40 - 50%, never 0%.

═══════════════════════════════════════════════════
STEP 3 — BUILD THREE KNOWLEDGE GRAPHS
═══════════════════════════════════════════════════
Build THREE separate KGs (variants), each anchored to its theme phrases from
STEP 1 and using its KEEP/REPLACE partition from STEP 2, with EXACTLY this
ontology. The three KGs must not be identical to each other.

RELATIONSHIPS — use only these:
  Theme-to-entity (the theme is always the HEAD, type CONCEPT):
    HAS_ACTOR | HAS_TARGET | HAS_CONTEXT | AFFECTS
  Entity-to-entity (support, one hop from the theme):
    ACTS_ON | BELONGS_TO

ENTITY TYPES — use exactly one per entity:
  ORG | ORG/GOV | ORG/REG | PERSON | GPE | COMP | PRODUCT | EVENT | SECTOR |
  ECON_INDICATOR | FIN_INSTRUMENT | CONCEPT

RULES:
  • CONCEPT nodes are ONLY the variant's theme phrases (near_paraphrase for
    variant 1, paraphrase for variants 2 and 3) — never invent others
  • Every triplet must connect to one of the variant's themes directly or in one hop
  • Max 4 words per entity label; no numeric or temporal entities
  • dimension values: economic_monetary_event | geopolitical_factor |
    sector_or_industry | policy_or_regulation | technology_concept | macro_trend

═══════════════════════════════════════════════════
OUTPUT — return ONLY valid JSON, no markdown fences, no extra text
═══════════════════════════════════════════════════
Each variant declares its KEEP set in "kept_entities" (original entity strings
copied verbatim); its knowledge_graph must actually use every one of them.
{
  "theme_map": [
    {"original": "<original theme>", "near_paraphrase": "<almost-same phrase>",
     "paraphrase": "<new phrase>", "dimension": "<dimension>"}
  ],
  "variants": [
    {
      "variant": 1,
      "kept_entities": ["<original entity kept verbatim>", ...],
      "knowledge_graph": [
        {"triplet": ["head", "head_type", "relationship", "tail", "tail_type"],
         "theme": "<near_paraphrase theme phrase>", "dimension": "<dimension>"}
      ]
    },
    {"variant": 2, "kept_entities": [...], "knowledge_graph": [... themes are the paraphrase phrases ...]},
    {"variant": 3, "kept_entities": [...], "knowledge_graph": [... themes are the paraphrase phrases ...]}
  ]
}
"""


def build_user_prompt(row) -> str:
    themes = json.loads(row["themes"])
    kg = json.loads(row["knowledge_graph"])
    return (
        "ORIGINAL ARTICLE DESCRIPTION:\n"
        + str(row.get("description", ""))[:600]
        + "\n\nORIGINAL DISCOVERED THEMES:\n"
        + json.dumps(themes, indent=1)
        + "\n\nORIGINAL KNOWLEDGE GRAPH:\n"
        + json.dumps(kg, indent=1)
    )

In [ ]:
OVERLAP_BANDS = {1: (0.60, 1.00), 2: (0.50, 0.95), 3: (0.20, 0.70)}


def extract_json(text):

    text = re.sub(r"```(?:json)?", "", text).strip()
    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("no JSON object in response")

    return json.loads(text[start : end + 1])


def raw_kg_entities(kg):

    entities = set()
    for t in kg if isinstance(kg, list) else []:
        triplet = t.get("triplet") if isinstance(t, dict) else None

        if not isinstance(triplet, (list, tuple)) or len(triplet) != 5:
            continue

        head, head_type, _, tail, tail_type = (str(x) for x in triplet)

        if head_type.upper() != "CONCEPT":
            entities.add(head.strip().lower())

        if tail_type.upper() != "CONCEPT":
            entities.add(tail.strip().lower())

    return entities


def check_overlap_bands(result, parent_entities, article_id):
    """raise ValueError (-> tenacity retry) if any variant's entity overlap with the original falls outside its band"""

    if not parent_entities:
        return

    for variant in result["variants"]:
        variant_num = variant.get("variant")

        if variant_num not in OVERLAP_BANDS:
            continue

        entities = raw_kg_entities(variant.get("knowledge_graph", []))
        overlap = len(entities & parent_entities) / len(parent_entities)
        low, high = OVERLAP_BANDS[variant_num]

        if not (low <= overlap <= high):
            raise ValueError(
                f"variant {variant_num} overlap {overlap:.2f} outside "
                f"[{low}, {high}] for article {article_id}"
            )

In [ ]:
@retry(
    retry=retry_if_exception_type((openai.RateLimitError, openai.APIError, ValueError)),
    wait=wait_exponential(multiplier=1, min=2, max=60),
    stop=stop_after_attempt(3),
)
def call_LLM(prompt, article_id, parent_entities):

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.8,
        max_tokens=MAX_TOKENS,
    )

    content = response.choices[0].message.content
    if not content:
        raise ValueError(f"Empty response for article {article_id}")

    result = extract_json(content)
    if not result.get("variants") or not result.get("theme_map"):
        raise ValueError(f"missing variants/theme_map for article {article_id}")

    check_overlap_bands(result, parent_entities, article_id)

    return {"article_id": article_id, "result": result}

In [ ]:
def clean_kg(kg, relax_concepts=False):

    cleaned = []
    seen = set()
    drops = Counter()

    # collect every theme phrase in article's KG
    article_themes = {
        e["theme"].strip().lower()
        for e in kg
        if isinstance(e, dict)
        and isinstance(e.get("theme"), str)
        and e["theme"].strip()
    }

    for entry in kg:

        if not isinstance(entry, dict):
            drops["not_dict"] += 1
            continue

        triplet = entry.get("triplet")
        theme = entry.get("theme")
        dimension = entry.get("dimension")

        # check basic shape
        if not isinstance(triplet, (list, tuple)) or len(triplet) != 5:
            drops["malformed_triplet"] += 1
            continue

        if not all(isinstance(x, str) and x.strip() for x in triplet):
            drops["empty_triplet"] += 1
            continue

        if not (isinstance(theme, str) and theme.strip()):
            drops["empty_theme"] += 1
            continue

        # extract triplet elements
        head, htype, rel, tail, ttype = (x.strip() for x in triplet)
        htype, ttype, rel = htype.upper(), ttype.upper(), rel.upper()
        dimension = str(dimension).strip().lower()
        theme = theme.strip()

        # check relation and dimension against the ontology
        if htype not in ENTITY_TYPES:
            drops["invalid_head_type"] += 1
            continue

        if ttype not in ENTITY_TYPES:
            drops["invalid_tail_type"] += 1
            continue

        if rel not in ALL_RELATIONS:
            drops["invalid_relation"] += 1
            continue

        if dimension not in DIMENSIONS:
            drops["invalid_dimension"] += 1
            continue

        # CONCEPT nodes are only allowed to be theme anchors
        valid_concepts = article_themes if relax_concepts else {theme.lower()}
        if rel in THEME_RELATIONS and "CONCEPT" not in (htype, ttype):
            drops["theme_rel_no_concept"] += 1
            continue
        if htype == "CONCEPT" and head.lower() not in valid_concepts:
            drops["concept_not_theme"] += 1
            continue
        if ttype == "CONCEPT" and tail.lower() not in valid_concepts:
            drops["concept_not_theme"] += 1
            continue

        # drop self loops and exact duplicates
        if head.lower() == tail.lower():
            drops["self_loop"] += 1
            continue

        key = (head.lower(), htype, rel, tail.lower(), ttype, theme.lower())
        if key in seen:
            drops["duplicate_triplet"] += 1
            continue

        seen.add(key)

        cleaned.append(
            {
                "triplet": [head, htype, rel, tail, ttype],
                "theme": theme,
                "dimension": dimension,
            }
        )

    return cleaned, drops


def extract_themes(kg):
    """unique (theme, dimension) pairs, keeping the order they first appear"""

    themes = []
    seen = set()
    for t in kg:
        pair = (t["theme"], t["dimension"])

        if pair not in seen:
            seen.add(pair)
            themes.append({"theme": t["theme"], "dimension": t["dimension"]})

    return themes


def kg_entities(kg):

    entities = set()
    for t in kg:
        head, head_type, _, tail, tail_type = t["triplet"]

        if head_type != "CONCEPT":
            entities.add(head.lower())

        if tail_type != "CONCEPT":
            entities.add(tail.lower())

    return entities

### Call LLM model

In [ ]:
src = pd.read_csv(SRC_CSV)

# filter for cleaned KG + one discovered theme
pool = src[src["knowledge_graph"].notna() & src["themes"].notna()].copy()
pool = pool[pool["themes"].apply(lambda t: len(json.loads(t)) > 0)]
pool = pool[pool["knowledge_graph"].apply(lambda k: len(json.loads(k)) > 0)]
print(f"Eligible originals: {len(pool):,}")

# skip originals we've already generated (they're in the ledger)
ledger = json.loads(LEDGER_PATH.read_text()) if LEDGER_PATH.exists() else []
done_ids = {r["parent_article_id"] for r in ledger}
unprocessed = pool[~pool["article_id"].isin(done_ids)]

# take a fresh random sample of not done originals for this run
todo = unprocessed.sample(n=min(RUN_SIZE, len(unprocessed)))

print(
    f"{len(done_ids)} already generated, {len(unprocessed)} unprocessed, {len(todo)} in this run"
)

Eligible originals: 5,793
233 already generated, 5560 unprocessed, 150 in this run


In [ ]:
failures = json.loads(FAILURES_PATH.read_text()) if FAILURES_PATH.exists() else []

# call LLM; append to the ledger as we go
for _, row in tqdm(todo.iterrows(), total=len(todo), unit="article"):
    try:
        parent_entities = kg_entities(json.loads(row["knowledge_graph"]))
        out = call_LLM(build_user_prompt(row), row["article_id"], parent_entities)[
            "result"
        ]
        ledger.append(
            {
                "parent_article_id": row["article_id"],
                "model": LLM_MODEL,
                "theme_map": out["theme_map"],
                "variants": out["variants"],
            }
        )
        LEDGER_PATH.write_text(json.dumps(ledger, indent=2))

    except Exception as e:

        failures.append(
            {
                "parent_article_id": row["article_id"],
                "error": f"{type(e).__name__}: {e}",
            }
        )
        FAILURES_PATH.write_text(json.dumps(failures, indent=2))

print(f"Ledger: {len(ledger)} originals done, failures: {len(failures)}")

  0%|          | 0/150 [00:00<?, ?article/s]

Ledger: 356 originals done, failures: 28


### Validate & clean synthetic KGs

In [ ]:
# for each synthetic: clean its KG, check the paraphrases, and measure entity overlap parent lookups come from the full eligible pool
parent_kg = {
    r["article_id"]: json.loads(r["knowledge_graph"]) for _, r in pool.iterrows()
}
parent_row = {r["article_id"]: r for _, r in pool.iterrows()}

records = []
total_drops = Counter()
bad_paraphrase = []
empty_after_clean = []

for record in ledger:
    pid = record["parent_article_id"]
    model = record.get("model", "")
    orig_entities = kg_entities(parent_kg.get(pid, []))

    # flag reworded phrases that are identical to the original theme (both the
    # near_paraphrase and the paraphrase must be distinct strings)
    for mapping in record["theme_map"]:
        orig = mapping["original"].strip().lower()
        for field in ("near_paraphrase", "paraphrase"):
            if mapping.get(field, "").strip().lower() == orig:
                bad_paraphrase.append((pid, field, mapping["original"]))

    for variant in record["variants"]:
        variant_num = variant.get("variant")
        synth_id = f"{pid}_syn{variant_num}"

        cleaned, drops = clean_kg(
            variant.get("knowledge_graph", []), relax_concepts=True
        )
        total_drops.update(drops)
        if not cleaned:
            empty_after_clean.append(synth_id)

        entities = kg_entities(cleaned)
        overlap = len(entities & orig_entities) / max(len(orig_entities), 1)
        records.append(
            {
                "synth_id": synth_id,
                "parent_article_id": pid,
                "model": model,
                "variant": variant_num,
                "themes": json.dumps(extract_themes(cleaned)),
                "knowledge_graph": json.dumps(cleaned),
                "n_triplets": len(cleaned),
                "entity_overlap": round(overlap, 3),
                "parent_themes": json.dumps(json.loads(parent_row[pid]["themes"])),
            }
        )

synth_df = pd.DataFrame(records)
print(f"Synthetics built: {len(synth_df)} (from {len(ledger)} originals)")
print(f"Identical (bad) paraphrases: {len(bad_paraphrase)}  {bad_paraphrase[:5]}")
print(f"Empty KG after cleaning: {len(empty_after_clean)}  {empty_after_clean[:5]}")
print("\nKG cleaning drops:")
for reason, count in total_drops.most_common():
    print(f"  {reason}: {count:,}")

# mean entity overlap per variant - the gradient should read
# syn1 ~0.8-0.9, syn2 ~0.7-0.8, syn3 ~0.4-0.5
print("\nMean entity overlap by variant (expect ~0.8-0.9 / ~0.7-0.8 / ~0.4-0.5):")
print(synth_df.groupby("variant")["entity_overlap"].mean().round(3).to_string())

Synthetics built: 1068 (from 356 originals)
Identical (bad) paraphrases: 4  [('f292ff658b26', 'near_paraphrase', 'smartphone market saturation'), ('ce7368c641c4', 'near_paraphrase', 'Oil Crisis'), ('4e518f8e8e52', 'near_paraphrase', 'Semiconductor Chips'), ('7560100742b4', 'near_paraphrase', 'Semiconductor Chips')]
Empty KG after cleaning: 0  []

KG cleaning drops:
  concept_not_theme: 3

Mean entity overlap by variant (expect ~0.8-0.9 / ~0.7-0.8 / ~0.4-0.5):
variant
1    0.873
2    0.727
3    0.459


### Assemble golden set + expected same cluster pairs

In [11]:
synth_df.to_csv(OUT_CSV, index=False)
print(f"Saved {len(synth_df)} synthetic KG variants -> {OUT_CSV}")

# Every item in a family (the parent + all its synthetic variants) should
# cluster together, so every unordered pair among them is an expected-same pair.
# Using combinations keeps this correct for any number of variants.
pairs = []
for pid, group in synth_df.groupby("parent_article_id"):
    # family members labelled: the parent, then syn1, syn2, syn3, ...
    model = group["model"].iloc[0]
    members = [(pid, "parent")]
    for _, s in group.sort_values("variant").iterrows():
        members.append((s["synth_id"], f"syn{s['variant']}"))

    for (id_a, label_a), (id_b, label_b) in combinations(members, 2):
        pairs.append(
            {
                "family_id": pid,
                "model": model,
                "id_a": id_a,
                "id_b": id_b,
                "pair_type": f"{label_a}-{label_b}",
            }
        )

pairs_df = pd.DataFrame(pairs)
pairs_df.to_csv(PAIRS_CSV, index=False)
print(f"Saved {len(pairs_df)} expected-same-cluster pairs -> {PAIRS_CSV}")
pairs_df.head(6)

Saved 1068 synthetic KG variants -> ../data/synthetic_golden_set.csv
Saved 2136 expected-same-cluster pairs -> ../data/golden_pairs.csv


,family_id,model,id_a,id_b,pair_type
0,000c3c398919,nvidia/nemotron-3-ultra-550b-a55b:free,000c3c398919,000c3c398919_syn1,parent-syn1
1,000c3c398919,nvidia/nemotron-3-ultra-550b-a55b:free,000c3c398919,000c3c398919_syn2,parent-syn2
2,000c3c398919,nvidia/nemotron-3-ultra-550b-a55b:free,000c3c398919,000c3c398919_syn3,parent-syn3
3,000c3c398919,nvidia/nemotron-3-ultra-550b-a55b:free,000c3c398919_syn1,000c3c398919_syn2,syn1-syn2
4,000c3c398919,nvidia/nemotron-3-ultra-550b-a55b:free,000c3c398919_syn1,000c3c398919_syn3,syn1-syn3
5,000c3c398919,nvidia/nemotron-3-ultra-550b-a55b:free,000c3c398919_syn2,000c3c398919_syn3,syn2-syn3


### Theme level gold mapping

In [12]:
# Build the theme-level gold set from theme_map. Each (original, near_paraphrase)
# and (original, paraphrase) pair is an edge, and each gold cluster is a connected
# group of theme phrases. We use union-find so that a theme string that shows up
# in several families still maps to one cluster id.
uf_parent = {}  # theme -> parent (union-find)
display_name = {}  # theme -> the string we display for it (first one seen)
theme_info = {}  # theme -> {role, dimension, family_id}


def find(x):
    uf_parent.setdefault(x, x)
    while uf_parent[x] != x:
        uf_parent[x] = uf_parent[uf_parent[x]]
        x = uf_parent[x]
    return x


def union(a, b):
    root_a, root_b = find(a), find(b)
    if root_a != root_b:
        uf_parent[root_a] = root_b


def remember_theme(phrase, role, dimension, family_id, model):
    key = phrase.strip().lower()
    display_name.setdefault(key, phrase.strip())
    # first writer wins for role/dimension/family, but an "original" role is
    # allowed to overwrite a non-original one
    if key not in theme_info or (
        theme_info[key]["role"] != "original" and role == "original"
    ):
        theme_info[key] = {
            "role": role,
            "dimension": dimension,
            "family_id": family_id,
            "model": model,
        }
    find(key)
    return key


# which variants each reworded phrase actually survived into (after clean_kg)
variants_present = {}  # theme -> set of variant numbers it appears in
for _, row in synth_df.iterrows():
    for t in json.loads(row["themes"]):
        variants_present.setdefault(t["theme"].strip().lower(), set()).add(
            int(row["variant"])
        )

# all variant numbers seen across the set (1, 2, 3, ...) -> one in_syn{k} column each
all_variants = sorted({int(row["variant"]) for _, row in synth_df.iterrows()})

# link every original theme to its near_paraphrase (variant 1) and paraphrase (variants 2-3)
for record in ledger:
    family_id = record["parent_article_id"]
    model = record.get("model", "")

    for mapping in record["theme_map"]:
        orig_key = remember_theme(
            mapping["original"],
            "original",
            mapping.get("dimension", ""),
            family_id,
            model,
        )
        near_key = remember_theme(
            mapping["near_paraphrase"],
            "near_paraphrase",
            mapping.get("dimension", ""),
            family_id,
            model,
        )
        para_key = remember_theme(
            mapping["paraphrase"],
            "paraphrase",
            mapping.get("dimension", ""),
            family_id,
            model,
        )
        union(orig_key, near_key)
        union(orig_key, para_key)

# give each cluster a stable id, numbered by first appearance
cluster_id = {}
rows = []
for key in theme_info:  # dict keeps first-seen order
    root = find(key)
    if root not in cluster_id:
        cluster_id[root] = f"gt_{len(cluster_id):04d}"
    meta = theme_info[key]
    row = {
        "theme": display_name[key],
        "gold_cluster_id": cluster_id[root],
        "model": meta["model"],
        "dimension": meta["dimension"],
        "role": meta["role"],
        "family_id": meta["family_id"],
    }
    for v in all_variants:
        row[f"in_syn{v}"] = v in variants_present.get(key, set())
    row["keep"] = 1
    rows.append(row)

gold_themes = pd.DataFrame(rows)

# every gold cluster should have at least 2 theme phrases (size 3 is the norm:
# original + near_paraphrase + paraphrase; 2 happens if two phrasings collide)
sizes = gold_themes["gold_cluster_id"].value_counts()
assert (sizes >= 2).all(), f"singleton gold clusters: {sizes[sizes < 2].index.tolist()}"

gold_themes.to_csv(THEME_GOLD_CSV, index=False)
print(
    f"Saved {len(gold_themes)} theme rows in {sizes.size} gold clusters -> {THEME_GOLD_CSV}"
)
print(
    f"cluster size distribution: {dict(Counter(sizes.values))}  (3 = original + near_paraphrase + paraphrase)"
)

# clusters bigger than 3 mean a theme phrase recurred across families
merged = sizes[sizes > 3]
if len(merged):
    print(f"\n{len(merged)} merged clusters (a theme phrase recurred across families):")
    for cid in merged.index[:10]:
        print(
            f"  {cid}: {gold_themes.loc[gold_themes.gold_cluster_id == cid, 'theme'].tolist()}"
        )
gold_themes.head(8)

Saved 3134 theme rows in 1002 gold clusters -> ../data/synthetic_gold_themes.csv
cluster size distribution: {np.int64(12): 1, np.int64(11): 1, np.int64(10): 2, np.int64(9): 3, np.int64(8): 3, np.int64(7): 4, np.int64(6): 2, np.int64(5): 19, np.int64(4): 7, np.int64(3): 957, np.int64(2): 3}  (3 = original + near_paraphrase + paraphrase)

42 merged clusters (a theme phrase recurred across families):
  gt_0264: ['Fed interest rates', 'Federal Reserve rates', 'central bank lending rates', 'Federal Reserve interest rates', 'central bank rate policy', 'Central Bank Interest Policy', 'central bank rate policies', 'monetary policy decisions', 'Central bank rates', 'central bank interest rate', 'monetary policy interest setting', 'US central bank borrowing costs']
  gt_0044: ['central bank monetary tightening', 'central bank tightening', 'monetary policy contraction', 'central bank rate tightening', 'rate tightening by central bank', 'monetary policy tightening', 'monetary policy interest hike'

,theme,gold_cluster_id,model,dimension,role,family_id,in_syn1,in_syn2,in_syn3,keep
0,subcontracted labor unionization,gt_0000,tencent/hy3:free,sector_or_industry,original,066f1e5fe45f,False,False,False,1
1,subcontracted worker unionization,gt_0000,tencent/hy3:free,sector_or_industry,near_paraphrase,066f1e5fe45f,True,False,False,1
2,outsourced staff organizing,gt_0000,tencent/hy3:free,sector_or_industry,paraphrase,066f1e5fe45f,False,True,True,1
3,joint employer dispute,gt_0001,tencent/hy3:free,policy_or_regulation,original,066f1e5fe45f,False,False,False,1
4,shared employer dispute,gt_0001,tencent/hy3:free,policy_or_regulation,near_paraphrase,066f1e5fe45f,True,False,False,1
5,co-employer legal conflict,gt_0001,tencent/hy3:free,policy_or_regulation,paraphrase,066f1e5fe45f,False,True,True,1
6,extreme heat workplace safety,gt_0002,tencent/hy3:free,sector_or_industry,original,066f1e5fe45f,False,False,False,1
7,severe heat job safety,gt_0002,tencent/hy3:free,sector_or_industry,near_paraphrase,066f1e5fe45f,True,False,False,1
